In [278]:
import tensorflow as tf

import numpy as np
import os
import time

https://www.tensorflow.org/text/tutorials/text_generation

In [279]:
path_to_file = tf.keras.utils.get_file('BorgeX - Corpus.txt', 'https://raw.githubusercontent.com/Duhkha-dev/BX/main/BorgeX%20-%20Corpus.txt')

In [280]:
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(f'Length of text: {len(text)} characters')

Length of text: 719634 characters


In [281]:
print(text[:250])

Debo a la conjunción de un espejo y de una
enciclopedia el descubrimiento de Uqbar. El
espejo inquietaba el fondo de un corredor en una
quinta de la calle Gaona, en Ramos Mejía; la
enciclopedia falazmente se llama The AngloAmerican Cyclopaedia (N


In [282]:
vocab = sorted(set(text))
print(f'{len(vocab)} unique characters')

112 unique characters


In [283]:
example_texts = ['abcdefg', 'xyz']

chars = tf.strings.unicode_split(example_texts, input_encoding='UTF-8')
chars

<tf.RaggedTensor [[b'a', b'b', b'c', b'd', b'e', b'f', b'g'], [b'x', b'y', b'z']]>

In [284]:
ids_from_chars = tf.keras.layers.StringLookup(
    vocabulary=list(vocab), mask_token=None)

In [285]:
ids = ids_from_chars(chars)
ids

<tf.RaggedTensor [[58, 59, 60, 61, 62, 63, 64], [81, 82, 83]]>

In [286]:
chars_from_ids = tf.keras.layers.StringLookup(
    vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)

In [287]:
chars = chars_from_ids(ids)
chars

<tf.RaggedTensor [[b'a', b'b', b'c', b'd', b'e', b'f', b'g'], [b'x', b'y', b'z']]>

In [288]:
tf.strings.reduce_join(chars, axis=-1).numpy()

array([b'abcdefg', b'xyz'], dtype=object)

In [289]:
def text_from_ids(ids):
  return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

In [290]:
all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
all_ids

<tf.Tensor: shape=(719634,), dtype=int64, numpy=array([32, 62, 59, ...,  1,  3,  1])>

In [291]:
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)

In [292]:
for ids in ids_dataset.take(10):
    print(chars_from_ids(ids).numpy().decode('utf-8'))

D
e
b
o
 
a
 
l
a
 


In [293]:
seq_length = 100
examples_per_epoch = len(text)//(seq_length+1)

In [294]:
sequences = ids_dataset.batch(seq_length+1, drop_remainder=True)

for seq in sequences.take(1):
  print(chars_from_ids(seq))

tf.Tensor(
[b'D' b'e' b'b' b'o' b' ' b'a' b' ' b'l' b'a' b' ' b'c' b'o' b'n' b'j'
 b'u' b'n' b'c' b'i' b'\xc3\xb3' b'n' b' ' b'd' b'e' b' ' b'u' b'n' b' '
 b'e' b's' b'p' b'e' b'j' b'o' b' ' b'y' b' ' b'd' b'e' b' ' b'u' b'n'
 b'a' b'\r' b'\n' b'e' b'n' b'c' b'i' b'c' b'l' b'o' b'p' b'e' b'd' b'i'
 b'a' b' ' b'e' b'l' b' ' b'd' b'e' b's' b'c' b'u' b'b' b'r' b'i' b'm'
 b'i' b'e' b'n' b't' b'o' b' ' b'd' b'e' b' ' b'U' b'q' b'b' b'a' b'r'
 b'.' b' ' b'E' b'l' b'\r' b'\n' b'e' b's' b'p' b'e' b'j' b'o' b' ' b'i'
 b'n' b'q' b'u' b'i'], shape=(101,), dtype=string)


In [295]:
for seq in sequences.take(5):
  print(text_from_ids(seq).numpy())

b'Debo a la conjunci\xc3\xb3n de un espejo y de una\r\nenciclopedia el descubrimiento de Uqbar. El\r\nespejo inqui'
b'etaba el fondo de un corredor en una\r\nquinta de la calle Gaona, en Ramos Mej\xc3\xada; la\r\nenciclopedia fala'
b'zmente se llama The AngloAmerican Cyclopaedia (New York, 1917) y es una\r\nreimpresi\xc3\xb3n literal, pero ta'
b'mbi\xc3\xa9n morosa, de la\r\nEncyclopaedia Britannica de 1902. El hecho se\r\nprodujo har\xc3\xa1 unos cinco a\xc3\xb1os. Bio'
b'y Casares hab\xc3\xada\r\ncenado conmigo esa noche y nos demor\xc3\xb3 una vasta\r\npol\xc3\xa9mica sobre la ejecuci\xc3\xb3n de una '


In [296]:
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

In [297]:
split_input_target(list("Tensorflow"))

(['T', 'e', 'n', 's', 'o', 'r', 'f', 'l', 'o'],
 ['e', 'n', 's', 'o', 'r', 'f', 'l', 'o', 'w'])

In [298]:
dataset = sequences.map(split_input_target)

In [299]:
for input_example, target_example in dataset.take(1):
    print("Input :", text_from_ids(input_example).numpy())
    print("Target:", text_from_ids(target_example).numpy())

Input : b'Debo a la conjunci\xc3\xb3n de un espejo y de una\r\nenciclopedia el descubrimiento de Uqbar. El\r\nespejo inqu'
Target: b'ebo a la conjunci\xc3\xb3n de un espejo y de una\r\nenciclopedia el descubrimiento de Uqbar. El\r\nespejo inqui'


In [300]:

BATCH_SIZE = 64

BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE))

dataset

<PrefetchDataset shapes: ((64, 100), (64, 100)), types: (tf.int64, tf.int64)>

In [301]:

vocab_size = len(vocab)

embedding_dim = 256

rnn_units = 1024

In [302]:
class MyModel(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__(self)
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(rnn_units,
                                   return_sequences=True,
                                   return_state=True)
    self.dense = tf.keras.layers.Dense(vocab_size)

  def call(self, inputs, states=None, return_state=False, training=False):
    x = inputs
    x = self.embedding(x, training=training)
    if states is None:
      states = self.gru.get_initial_state(x)
    x, states = self.gru(x, initial_state=states, training=training)
    x = self.dense(x, training=training)

    if return_state:
      return x, states
    else:
      return x

In [303]:
model = MyModel(
    vocab_size=len(ids_from_chars.get_vocabulary()),
    embedding_dim=embedding_dim,
    rnn_units=rnn_units)

In [304]:
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")

(64, 100, 113) # (batch_size, sequence_length, vocab_size)


In [305]:
model.summary()

Model: "my_model_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_10 (Embedding)    multiple                  28928     
                                                                 
 gru_10 (GRU)                multiple                  3938304   
                                                                 
 dense_10 (Dense)            multiple                  115825    
                                                                 
Total params: 4,083,057
Trainable params: 4,083,057
Non-trainable params: 0
_________________________________________________________________


In [306]:
sampled_indices = tf.random.categorical(example_batch_predictions[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices, axis=-1).numpy()

In [307]:
sampled_indices

array([ 27,  65,  83,  53,  38,   0,  20,  25,  77,   7,   3, 107, 109,
        29,  65,  58,  81,   6,  89,  70, 100,  46,  93,  66,  34,  13,
        41,  92,  59,   4,  56, 101,  55,   8,  37,  27,   9,  52,  66,
        23,  73,  29,  76,  94,  22,  65,  93,  29,  11,  91, 110,  77,
        62,  81,  25,   6,  65,  42,  67,  78,  51,  64,  70,  16, 103,
        42, 108,  86, 109, 101,  92,  81,  40,  44,  62,  47,  44,  44,
        35,  42,  49,   1,  24,  22,  28,  74,  64,  28,  31,  34,  31,
        31,  62, 108,  47,  54,  62,  95,  75,  54])

In [308]:
print("Input:\n", text_from_ids(input_example_batch[0]).numpy())
print()
print("Next Char Predictions:\n", text_from_ids(sampled_indices).numpy())

Input:
 b'reso no pod\xc3\xada prescindir de una biblioteca\r\nde libros de consulta; Nierenstein, que trabajaba en una'

Next Char Predictions:
 b'>hzYJ[UNK]6;t\'\r\xc3\xbc\xe2\x80\x99Ahax"\xc3\x81m\xc3\xadR\xc3\x9aiF.M\xc3\x91b \\\xc3\xae[(I>)Xi9pAs\xc3\xa08h\xc3\x9aA,\xc3\x8d\xe2\x80\x9ctex;"hNjuWgm2\xc3\xb3N\xe2\x80\x94\xc2\xba\xe2\x80\x99\xc3\xae\xc3\x91xLPeSPPGNU\n:8?qg?CFCCe\xe2\x80\x94SZe\xc3\xa1rZ'


In [309]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)

In [310]:
example_batch_loss = loss(target_example_batch, example_batch_predictions)
mean_loss = example_batch_loss.numpy().mean()
print("Prediction shape: ", example_batch_predictions.shape, " # (batch_size, sequence_length, vocab_size)")
print("Mean loss:        ", mean_loss)

Prediction shape:  (64, 100, 113)  # (batch_size, sequence_length, vocab_size)
Mean loss:         4.7267475


In [311]:
tf.exp(mean_loss).numpy()

112.92767

In [312]:
model.compile(optimizer='adam', loss=loss)

In [313]:

checkpoint_dir = './training_checkpoints'

checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix,
    save_weights_only=True)

In [314]:
EPOCHS = 30

In [315]:
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

Epoch 1/30
111/111 [==============================] - 19s 138ms/step - loss: 2.8123
Epoch 2/30
111/111 [==============================] - 16s 136ms/step - loss: 2.1771
Epoch 3/30
111/111 [==============================] - 16s 136ms/step - loss: 2.0065
Epoch 4/30
111/111 [==============================] - 16s 136ms/step - loss: 1.8619
Epoch 5/30
111/111 [==============================] - 16s 136ms/step - loss: 1.7237
Epoch 6/30
111/111 [==============================] - 16s 136ms/step - loss: 1.6060
Epoch 7/30
111/111 [==============================] - 16s 136ms/step - loss: 1.5120
Epoch 8/30
111/111 [==============================] - 16s 137ms/step - loss: 1.4343
Epoch 9/30
111/111 [==============================] - 16s 137ms/step - loss: 1.3681
Epoch 10/30
111/111 [==============================] - 16s 137ms/step - loss: 1.3098
Epoch 11/30
111/111 [==============================] - 16s 137ms/step - loss: 1.2553
Epoch 12/30
111/111 [==============================] - 16s 137ms/step - lo

In [316]:
class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature = temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    # Create a mask to prevent "[UNK]" from being generated.
    skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        # Put a -inf at each bad index.
        values=[-float('inf')]*len(skip_ids),
        indices=skip_ids,
        # Match the shape to the vocabulary
        dense_shape=[len(ids_from_chars.get_vocabulary())])
    self.prediction_mask = tf.sparse.to_dense(sparse_mask)

  @tf.function
  def generate_one_step(self, inputs, states=None):
    # Convert strings to token IDs.
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Run the model.
    # predicted_logits.shape is [batch, char, next_char_logits]
    predicted_logits, states = self.model(inputs=input_ids, states=states,
                                          return_state=True)
    # Only use the last prediction.
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    # Apply the prediction mask: prevent "[UNK]" from being generated.
    predicted_logits = predicted_logits + self.prediction_mask

    # Sample the output logits to generate token IDs.
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    # Convert from token ids to characters
    predicted_chars = self.chars_from_ids(predicted_ids)

    # Return the characters and model state.
    return predicted_chars, states

In [317]:
one_step_model = OneStep(model, chars_from_ids, ids_from_chars)

In [318]:
start = time.time()
states = None
next_char = tf.constant(['Pienso que '])
result = [next_char]

for n in range(200):
  next_char, states = one_step_model.generate_one_step(next_char, states=states)
  result.append(next_char)

result = tf.strings.join(result)
end = time.time()
print(result[0].numpy().decode('utf-8'), '\n\n' + '_'*80)
print('\nRun time:', end - start)

Pienso que mi parece cantando y le pregunté si versificar a un patio de
freche; el día considerado redestorró con ansiedad,
cuyas pirámides en las sagas de Islandia.
Enmendó asimismo vasos una explicación
po 

________________________________________________________________________________

Run time: 1.9233510494232178
